In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sys

In [2]:
path="/sortref/system/rea/glo_bgc/microrys12/microrys3/GLOBAL_MULTIYEAR_BGC_001_033/cmems_mod_glo_bgc_my_0.083deg-lmtl_P1D-i/1998/01/cmems_mod_glo_bgc_my_0.083deg-lmtl_P1D-i_19980101.nc"
ds= xr.open_dataset(path)
ds

<xarray.Dataset> Size: 635MB
Dimensions:       (time: 1, latitude: 2040, longitude: 4320)
Coordinates:
  * time          (time) datetime64[ns] 8B 1998-01-01T12:00:00
  * latitude      (latitude) float32 8kB -80.0 -79.92 -79.83 ... 89.83 89.92
  * longitude     (longitude) float32 17kB -180.0 -179.9 -179.8 ... 179.8 179.9
Data variables:
    zooc          (time, latitude, longitude) float64 71MB ...
    mnkc_epi      (time, latitude, longitude) float64 71MB ...
    mnkc_umeso    (time, latitude, longitude) float64 71MB ...
    mnkc_mumeso   (time, latitude, longitude) float64 71MB ...
    mnkc_lmeso    (time, latitude, longitude) float64 71MB ...
    mnkc_mlmeso   (time, latitude, longitude) float64 71MB ...
    mnkc_hmlmeso  (time, latitude, longitude) float64 71MB ...
    zeu           (time, latitude, longitude) float64 71MB ...
    npp           (time, latitude, longitude) float64 71MB ...
Attributes: (12/14)
    title:                            Global ocean low and mid trophic levels...
    source:                           SEAPODYM-LMTL 3.0.0
    native_grid:                      Yin-Yang overset grid
    references:                       http://www.cls.fr; http://www.seapodym.eu
    institution:                      CLS
    source_physical_variables:        GLOBAL_REANALYSIS_PHY_001_030 CMEMS pro...
    ...                               ...
    date_field:                       1998-01-01
    spatial_resolution:               0.083x0.083
    temporal_resolution:              1 day
    domain:                           global
    history:                          Created on 2024-10-16; Updated micronek...
    Conventions:                      CF-1.8

## Moyenne mensuelle des données journalières

In [3]:
root = Path("/sortref/system/rea/glo_bgc/microrys12/microrys3/GLOBAL_MULTIYEAR_BGC_001_033/cmems_mod_glo_bgc_my_0.083deg-lmtl_P1D-i")
output = Path("/data/rd_exchange/asauvebois/moy_month")
output.mkdir(parents=True, exist_ok=True)

In [ ]:
for year_dir in sorted(root.iterdir()):
    if not year_dir.is_dir():
        continue
    print(f"\nAnnée : {year_dir.name}")
    
    for month_dir in sorted(year_dir.iterdir()):
        if not month_dir.is_dir():
            continue
        print(f"   Mois : {month_dir.name}")
        
        outfile = output/ f"{year_dir.name}_{month_dir.name}_monthly_mean.nc"  # ← ici
        
        if outfile.exists():
            print(f"   → déjà traité, on passe")
            continue
        
        files = sorted(month_dir.glob("*.nc"))
        if len(files) == 0:
            continue
        
        ds = xr.open_mfdataset(
            files,
            combine="by_coords",
            chunks={} 
        )
        monthly_mean = ds.mean(dim="time")
        monthly_mean.to_netcdf(outfile)
        ds.close()


Année : 1998
   Mois : 01
   → déjà traité, on passe
   Mois : 02
   → déjà traité, on passe
   Mois : 03
   → déjà traité, on passe
   Mois : 04
   → déjà traité, on passe
   Mois : 05
   → déjà traité, on passe
   Mois : 06
   → déjà traité, on passe
   Mois : 07
   → déjà traité, on passe
   Mois : 08
   → déjà traité, on passe
   Mois : 09
   → déjà traité, on passe
   Mois : 10
   → déjà traité, on passe
   Mois : 11
   → déjà traité, on passe
   Mois : 12
   → déjà traité, on passe

Année : 1999
   Mois : 01
   → déjà traité, on passe
   Mois : 02
   → déjà traité, on passe
   Mois : 03
   → déjà traité, on passe
   Mois : 04
   → déjà traité, on passe
   Mois : 05
   → déjà traité, on passe
   Mois : 06
   → déjà traité, on passe
   Mois : 07
   → déjà traité, on passe
   Mois : 08
   → déjà traité, on passe
   Mois : 09
   → déjà traité, on passe
   Mois : 10
   → déjà traité, on passe
   Mois : 11
   → déjà traité, on passe
   Mois : 12
   → déjà traité, on passe

Année : 200

## Vérification des valeurs des moyennes calculées


In [ ]:
root2 = Path("")

In [ ]:
annee=2000
mois= 11
lat_cible= 89.83
lon_cible= 179.8
variable= "mnkc_epi"

In [ ]:
month_dir = root/annee/mois
outfile = root2 / f"{annee}_{mois}_monthly_mean.nc"

files = sorted(month_dir.glob("*.nc"))

if len(files) == 0:
    print("Aucun fichier trouvé !")
else:
    # Fichiers journaliers
    ds = xr.open_mfdataset(files, combine="by_coords")
    serie = ds[variable].sel(
        latitude=lat_cible,
        longitude=lon_cible,
        method='nearest'
    )
    moy_month_test = serie.mean(dim='time').values

    # Fichier mensuel déjà calculé
    ds2 = xr.open_dataset(outfile)  # open_dataset car c'est un seul fichier
    serie2 = ds2[variable].sel(
        latitude=lat_cible,
        longitude=lon_cible,
        method='nearest'
    )
    moy_month_model = serie2.values

    print(f"Moyenne calculée depuis journaliers : {moy_month_test}")
    print(f"Moyenne depuis fichier mensuel      : {moy_month_model}")